# Linear Algebra in Deep Learning

深度学习中的线性代数综合应用。把前 7 节的知识串联起来，看线性代数如何支撑深度学习的每一个核心组件。

## 0. 环境配置与导入

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
print("PyTorch version:", torch.__version__)
torch.manual_seed(42)

## 1. 线性代数在深度学习中的全景图

深度学习的几乎所有运算都可以归结为线性代数运算：

| 深度学习组件 | 核心线性代数运算 | 对应课程章节 |
|-------------|----------------|------------|
| 全连接层 | 矩阵乘法 $Y = XW + b$ | 02 |
| 卷积层 | im2col + 矩阵乘法 / Toeplitz 矩阵 | 02, 05 |
| 注意力机制 | QK^T / softmax / V 的矩阵运算 | 02, 06 |
| 归一化 | 均值/方差计算、投影到单位球 | 05, 06 |
| 优化 | 梯度、Hessian、条件数、动量 | 03, 07 |
| 降维/表示 | PCA（SVD）、词嵌入几何 | 04, 05 |
| 初始化 | 正交矩阵、特征值谱 | 03, 04 |
| 正则化 | 岭回归（L2）、谱范数正则 | 06, 07 |
| 生成模型 | 协方差矩阵、Cholesky 采样 | 07 |

**核心洞察**：深度学习 = 线性变换 + 非线性激活的交替堆叠。理解线性代数，就理解了深度学习的骨架。

## 2. 全连接层：矩阵乘法的前向/反向/参数计数

### 2.1 前向传播

全连接层（Linear Layer）：$Y = XW^T + b$

- 输入 $X \in \mathbb{R}^{N \times D_{in}}$（N 个样本，D_in 维特征）
- 权重 $W \in \mathbb{R}^{D_{out} \times D_{in}}$
- 偏置 $b \in \mathbb{R}^{D_{out}}$
- 输出 $Y \in \mathbb{R}^{N \times D_{out}}$

### 2.2 反向传播

设上游梯度 $\frac{\partial L}{\partial Y} = dY \in \mathbb{R}^{N \times D_{out}}$：

$$
\frac{\partial L}{\partial W} = dY^T X \quad (D_{out} \times D_{in})
$$
$$
\frac{\partial L}{\partial b} = \sum_{i=1}^N dY_i \quad (D_{out})
$$
$$
\frac{\partial L}{\partial X} = dY W \quad (N \times D_{in})
$$

### 2.3 参数计数与计算量

- 参数数量：$D_{in} \times D_{out} + D_{out}$
- 乘法次数：$N \times D_{in} \times D_{out}$

In [ ]:
# 全连接层的前向和反向（手动实现，不用 nn.Linear）
torch.manual_seed(42)
N, D_in, D_out = 4, 5, 3

# 随机输入和权重
X = torch.randn(N, D_in)
W = torch.randn(D_out, D_in) * 0.1  # 小初始化
b = torch.zeros(D_out)

print(f"输入 X: {X.shape}")
print(f"权重 W: {W.shape}")
print(f"偏置 b: {b.shape}")

# 前向传播：Y = X @ W^T + b
Y = X @ W.T + b
print(f"\n前向输出 Y: {Y.shape}")
print("Y =\n", Y)

# 假设上游梯度 dY（比如从 MSE loss 来）
dY = torch.randn(N, D_out) * 0.01
print(f"\n上游梯度 dY: {dY.shape}")

# 反向传播
dW = dY.T @ X           # [D_out, D_in]
db = dY.sum(dim=0)      # [D_out]
dX = dY @ W              # [N, D_in]

print(f"\ndW: {dW.shape} (= dY^T @ X)")
print(f"db: {db.shape} (= dY.sum)")
print(f"dX: {dX.shape} (= dY @ W)")

# 验证：与 PyTorch autograd 一致
X2 = X.clone().requires_grad_(True)
W2 = W.clone().requires_grad_(True)
b2 = b.clone().requires_grad_(True)
Y2 = X2 @ W2.T + b2
Y2.backward(dY)
print(f"\n--- 与 autograd 对比 ---")
print(f"dW 一致? {torch.allclose(dW, W2.grad, atol=1e-5)}")
print(f"db 一致? {torch.allclose(db, b2.grad, atol=1e-5)}")
print(f"dX 一致? {torch.allclose(dX, X2.grad, atol=1e-5)}")

In [ ]:
# 参数计数与计算量分析
print("全连接层参数计数与计算量:")
print(f"{'D_in':>6} {'D_out':>6} {'参数量':>12} {'乘法次数(N=32)':>16}")
print("-" * 45)
for d_in, d_out in [(784, 256), (256, 128), (128, 10), (512, 512), (768, 3072)]:
    n_params = d_in * d_out + d_out
    n_mults = 32 * d_in * d_out
    print(f"{d_in:>6} {d_out:>6} {n_params:>12,} {n_mults:>16,}")

print("\n--- 典型网络的参数分布 ---")
# 一个简单的 MLP：784 -> 256 -> 128 -> 10
layers = [(784, 256), (256, 128), (128, 10)]
total = 0
for i, (d_in, d_out) in enumerate(layers):
    n = d_in * d_out + d_out
    total += n
    print(f"  层{i+1}: {d_in}->{d_out}, 参数 = {n:,}")
print(f"  总计: {total:,} 参数")
print(f"\n→ 全连接层的参数主要来自 W，b 只占很小比例")
print(f"→ 计算量与样本数 N、输入维度、输出维度三者成正比")

## 3. 卷积的矩阵视角（im2col）

### 3.1 卷积 = 特殊的矩阵乘法

2D 卷积本质上是**局部连接 + 权值共享**的线性变换。通过 **im2col** 技术，可以将卷积运算转化为标准矩阵乘法：

1. 将输入图像中每个卷积窗口展开为一行（im2col）
2. 将卷积核展开为一列
3. 做矩阵乘法
4. reshape 回输出特征图

### 3.2 为什么这样做？

- 复用高度优化的 BLAS/GEMM 矩阵乘法库
- 一次内存访问，计算效率高
- 是 cuDNN 等深度学习库的标准实现方式

### 3.3 Toeplitz 矩阵视角

对于 1D 卷积，卷积运算可以表示为 Toeplitz 矩阵（常数对角线矩阵）与向量的乘法。2D 卷积则是双块 Toeplitz 矩阵。

In [ ]:
# im2col 手动实现：将卷积转化为矩阵乘法
def im2col(input_data, kernel_h, kernel_w, stride=1, padding=0):
    """简单的 im2col 实现：将卷积窗口展开为列"""
    N, C, H, W = input_data.shape
    out_h = (H + 2*padding - kernel_h) // stride + 1
    out_w = (W + 2*padding - kernel_w) // stride + 1
    
    # padding
    if padding > 0:
        input_data = F.pad(input_data, (padding, padding, padding, padding))
    
    cols = []
    for i in range(out_h):
        for j in range(out_w):
            # 提取每个卷积窗口
            patch = input_data[:, :, i*stride:i*stride+kernel_h, j*stride:j*stride+kernel_w]
            cols.append(patch.reshape(N, -1))
    
    # cols: [N, out_h*out_w, C*kernel_h*kernel_w]
    cols = torch.stack(cols, dim=1)
    return cols, out_h, out_w

# 测试：1 张 1 通道 4x4 图像，3x3 卷积核，stride=1, padding=0
torch.manual_seed(42)
x = torch.randn(1, 1, 4, 4)  # [N, C, H, W]
w = torch.randn(1, 1, 3, 3)   # [out_C, in_C, kH, kW]
b = torch.randn(1)

print("输入 x:", x.shape)
print("卷积核 w:", w.shape)

# 方法1：PyTorch 官方 conv2d
y_torch = F.conv2d(x, w, b, stride=1, padding=0)
print(f"\n官方 conv2d 输出: {y_torch.shape}")
print("y_torch =\n", y_torch.squeeze())

# 方法2：im2col + 矩阵乘法
cols, out_h, out_w = im2col(x, 3, 3, stride=1, padding=0)
print(f"\nim2col 后 cols: {cols.shape} = [N, out_h*out_w, C*kH*kW]")
print(f"out_h={out_h}, out_w={out_w}")

# 卷积核展开为 [C*kH*kW, out_C]
w_col = w.reshape(w.shape[0], -1).T  # [9, 1]
print(f"卷积核展开 w_col: {w_col.shape}")

# 矩阵乘法：[N, out_h*out_w, 9] @ [9, 1] -> [N, out_h*out_w, 1]
y_col = cols @ w_col + b  # 加偏置
y_im2col = y_col.reshape(1, 1, out_h, out_w)

print(f"\nim2col 输出: {y_im2col.shape}")
print("y_im2col =\n", y_im2col.squeeze())
print(f"\n两种方法结果一致? {torch.allclose(y_torch, y_im2col, atol=1e-5)}")

In [ ]:
# 1D 卷积 = Toeplitz 矩阵 × 向量
def conv1d_toeplitz(kernel, signal_length):
    """将 1D 卷积转化为 Toeplitz 矩阵"""
    k = len(kernel)
    out_length = signal_length - k + 1
    # 构造 Toeplitz 矩阵
    T = torch.zeros(out_length, signal_length)
    for i in range(out_length):
        T[i, i:i+k] = kernel
    return T

# 测试
kernel = torch.tensor([1.0, 2.0, 3.0])  # 卷积核长度 3
signal = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])  # 信号长度 5

print("卷积核:", kernel.tolist())
print("信号:", signal.tolist())

# 方法1：直接卷积（valid mode）
y_direct = F.conv1d(signal.unsqueeze(0).unsqueeze(0), kernel.unsqueeze(0).unsqueeze(0)).squeeze()
print(f"\n直接卷积结果: {y_direct.tolist()}")

# 方法2：Toeplitz 矩阵
T = conv1d_toeplitz(kernel, len(signal))
print(f"\nToeplitz 矩阵 T ({T.shape}):")
print(T)
y_toeplitz = T @ signal
print(f"\nT @ signal = {y_toeplitz.tolist()}")
print(f"两种方法一致? {torch.allclose(y_direct, y_toeplitz)}")

print("\n→ 卷积的本质是结构化的矩阵乘法（Toeplitz/双块Toeplitz）")
print("→ 权值共享 = 矩阵中大量重复元素")
print("→ 局部连接 = 矩阵是稀疏的带状结构")

## 4. 注意力机制中的矩阵运算

### 4.1 Scaled Dot-Product Attention

Transformer 的核心运算：

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

涉及的线性代数运算：
1. $QK^T$：矩阵乘法，得到注意力分数矩阵 $[N, N]$
2. 除以 $\sqrt{d_k}$：标量缩放（控制方差，防止 softmax 饱和）
3. softmax：行归一化（非线性，但基于矩阵运算）
4. 结果 × V：矩阵乘法 $[N, N] \times [N, d_v]$

### 4.2 为什么除以 $\sqrt{d_k}$？

当 $d_k$ 很大时，$QK^T$ 的元素方差约为 $d_k$，导致值很大，softmax 进入饱和区（梯度接近 0）。除以 $\sqrt{d_k}$ 将方差归一化为 1。

### 4.3 多头注意力

多头 = 将 Q/K/V 按头数拆分，分别做注意力，再拼接。每头的维度 $d_k = d_{model} / h$。

In [ ]:
# 手动实现 Scaled Dot-Product Attention
torch.manual_seed(42)
N = 5       # 序列长度
d_model = 8 # 模型维度
d_k = d_model  # Q/K 维度

# 随机 Q, K, V
Q = torch.randn(N, d_k)
K = torch.randn(N, d_k)
V = torch.randn(N, d_k)

print(f"Q: {Q.shape}, K: {K.shape}, V: {V.shape}")

# Step 1: Q @ K^T -> 注意力分数
scores = Q @ K.T  # [N, N]
print(f"\n1. Q @ K^T = scores: {scores.shape}")
print("scores (前3行前3列):\n", scores[:3, :3])

# 方差分析
print(f"\n   scores 的方差: {scores.var().item():.4f}")
print(f"   理论方差 ≈ d_k = {d_k} (Q和K元素方差=1时)")

# Step 2: 缩放
scores_scaled = scores / (d_k ** 0.5)
print(f"\n2. 缩放后 scores / sqrt(d_k):")
print(f"   缩放后方差: {scores_scaled.var().item():.4f} (≈1)")

# Step 3: softmax（行归一化）
attn_weights = F.softmax(scores_scaled, dim=-1)
print(f"\n3. softmax 后注意力权重: {attn_weights.shape}")
print("   每行和 = 1?", torch.allclose(attn_weights.sum(dim=-1), torch.ones(N), atol=1e-5))
print("   注意力权重 (前3行):\n", attn_weights[:3])

# Step 4: 加权求和 V
output = attn_weights @ V  # [N, d_k]
print(f"\n4. output = attn_weights @ V: {output.shape}")
print("output (前3行):\n", output[:3])

# 验证：与 PyTorch 官方实现一致
output_torch = F.scaled_dot_product_attention(
    Q.unsqueeze(0), K.unsqueeze(0), V.unsqueeze(0)
).squeeze(0)
print(f"\n与官方 scaled_dot_product_attention 一致? {torch.allclose(output, output_torch, atol=1e-5)}")

In [ ]:
# 为什么要除以 sqrt(d_k)：softmax 饱和分析
d_k_values = [4, 16, 64, 256]
N = 10

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

unscaled_vals = []
scaled_vals = []
for d_k in d_k_values:
    Q = torch.randn(N, d_k)
    K = torch.randn(N, d_k)
    
    # 不缩放
    scores_unscaled = Q @ K.T
    attn_unscaled = F.softmax(scores_unscaled, dim=-1)
    max_attn_unscaled = attn_unscaled.max(dim=-1).values.mean().item()
    
    # 缩放
    scores_scaled = scores_unscaled / (d_k ** 0.5)
    attn_scaled = F.softmax(scores_scaled, dim=-1)
    max_attn_scaled = attn_scaled.max(dim=-1).values.mean().item()
    
    unscaled_vals.append(max_attn_unscaled)
    scaled_vals.append(max_attn_scaled)
    axes[0].scatter(d_k, max_attn_unscaled, s=100, zorder=5)
    axes[1].scatter(d_k, max_attn_scaled, s=100, zorder=5, color='red')

axes[0].plot(d_k_values, unscaled_vals, 'b--', alpha=0.5)
axes[0].set_xlabel("d_k (Q/K 维度)")
axes[0].set_ylabel("平均最大注意力权重")
axes[0].set_title("不缩放：d_k 越大，softmax 越饱和（接近 one-hot）")
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.1)

axes[1].plot(d_k_values, scaled_vals, 'r--', alpha=0.5)
axes[1].set_xlabel("d_k (Q/K 维度)")
axes[1].set_ylabel("平均最大注意力权重")
axes[1].set_title("缩放后：除以 sqrt(d_k)，各维度下分布均匀")
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1.1)

plt.suptitle("Scaled Dot-Product Attention：除以 sqrt(d_k) 防止 softmax 饱和", fontsize=13)
plt.tight_layout()
plt.show()

print("→ 不缩放时，d_k 大 → scores 方差大 → softmax 输出接近 one-hot → 梯度消失")
print("→ 缩放后，scores 方差≈1 → softmax 输出分布均匀 → 梯度正常")

In [ ]:
# 多头注意力：拆分 -> 分别注意力 -> 拼接
torch.manual_seed(42)
N = 6
d_model = 8
n_heads = 2
d_k = d_model // n_heads  # 每头维度 = 4

Q = torch.randn(N, d_model)
K = torch.randn(N, d_model)
V = torch.randn(N, d_model)

print(f"d_model={d_model}, n_heads={n_heads}, d_k(每头)={d_k}")
print(f"Q: {Q.shape}")

# 按头拆分：[N, d_model] -> [n_heads, N, d_k]
Q_heads = Q.reshape(N, n_heads, d_k).permute(1, 0, 2)  # [2, 6, 4]
K_heads = K.reshape(N, n_heads, d_k).permute(1, 0, 2)
V_heads = V.reshape(N, n_heads, d_k).permute(1, 0, 2)

print(f"\n拆分后 Q_heads: {Q_heads.shape} = [n_heads, N, d_k]")

# 每头分别做注意力
outputs = []
for h in range(n_heads):
    scores = Q_heads[h] @ K_heads[h].T / (d_k ** 0.5)
    attn = F.softmax(scores, dim=-1)
    out_h = attn @ V_heads[h]  # [N, d_k]
    outputs.append(out_h)
    print(f"  头{h}: scores {scores.shape}, output {out_h.shape}")

# 拼接：[n_heads, N, d_k] -> [N, n_heads*d_k] = [N, d_model]
multi_head_output = torch.cat(outputs, dim=-1)
print(f"\n拼接后 output: {multi_head_output.shape} = [N, d_model]")

# 验证：与 reshape 方式一致
# 标准实现：[N, d_model] -> [N, n_heads, d_k] -> 注意力 -> [N, n_heads, d_k] -> [N, d_model]
print(f"多头注意力本质：{n_heads} 个独立的小注意力并行计算，最后拼接")
print(f"每头只关注 d_k={d_k} 维的子空间，不同头学习不同的注意力模式")

## 5. 归一化层的线性代数

### 5.1 LayerNorm

LayerNorm 对每个样本的特征维度做归一化：

$$
\text{LayerNorm}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta
$$

其中 $\mu$ 和 $\sigma^2$ 是每个样本在特征维度上的均值和方差。

**线性代数视角**：LayerNorm 将每个样本向量投影到**均值为 0、范数为 $\sqrt{d}$** 的超平面上，然后再缩放和平移。

### 5.2 BatchNorm

BatchNorm 对每个特征维度在 batch 维度上做归一化：

$$
\text{BatchNorm}(x) = \gamma \odot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta
$$

**区别**：LayerNorm 沿特征维归一化（每个样本独立），BatchNorm 沿 batch 维归一化（每个特征独立）。

In [ ]:
# LayerNorm 的线性代数视角：投影到均值为0的超平面
torch.manual_seed(42)
N, d = 4, 8
x = torch.randn(N, d) * 2 + 1  # 均值≈1，方差≈4

print("输入 x:", x.shape)
print("每个样本的均值:", x.mean(dim=-1).tolist())
print("每个样本的方差:", x.var(dim=-1, unbiased=False).tolist())

# 手动 LayerNorm
mu = x.mean(dim=-1, keepdim=True)       # [N, 1]
sigma2 = x.var(dim=-1, keepdim=True, unbiased=False)  # [N, 1]
x_norm = (x - mu) / torch.sqrt(sigma2 + 1e-5)

# 可学习参数 gamma, beta（初始化为 1, 0）
gamma = torch.ones(d)
beta = torch.zeros(d)
y = gamma * x_norm + beta

print(f"\n归一化后:")
print("  均值:", y.mean(dim=-1).tolist(), "(≈0)")
print("  方差:", y.var(dim=-1, unbiased=False).tolist(), "(≈1)")

# 验证：与 PyTorch LayerNorm 一致
ln = torch.nn.LayerNorm(d, elementwise_affine=True)
ln.weight.data = gamma
ln.bias.data = beta
y_torch = ln(x)
print(f"\n与官方 LayerNorm 一致? {torch.allclose(y, y_torch, atol=1e-5)}")

# 几何可视化：2D 中 LayerNorm = 投影到直线 x1+x2=0 再缩放
print("\n--- 2D 几何解释 ---")
x_2d = torch.tensor([[3.0, 1.0]])  # 均值=2
mu_2d = x_2d.mean()
x_centered = x_2d - mu_2d  # [1, -1]，在直线 x1+x2=0 上
print(f"x = {x_2d.tolist()}, 均值 = {mu_2d.item()}")
print(f"x - 均值 = {x_centered.tolist()} (在超平面 sum(x)=0 上)")
print(f"归一化后 = {((x_2d - mu_2d) / x_centered.std()).tolist()}")
print("→ LayerNorm = 减去均值(投影到零和超平面) + 除以标准差(归一化范数)")

## 6. 优化中的线性代数

### 6.1 梯度与 Hessian

对于损失函数 $f(w)$：
- **梯度** $\nabla f(w)$：一阶导数，指向函数增长最快的方向
- **Hessian** $\nabla^2 f(w)$：二阶导数矩阵，描述函数的局部曲率

在极小值点附近，$f(w) \approx f(w^*) + \frac{1}{2}(w-w^*)^T H (w-w^*)$，其中 H 是 Hessian（正定矩阵）。

### 6.2 条件数与收敛速度

梯度下降的收敛速度由 Hessian 的条件数 $\kappa = \lambda_{max}/\lambda_{min}$ 决定：
- $\kappa \approx 1$：收敛快（各方向曲率均匀）
- $\kappa \gg 1$：收敛慢（锯齿状，高曲率方向震荡，低曲率方向缓慢）

### 6.3 动量（Momentum）

动量方法积累历史梯度，类似于物理中的惯性：
$$
v_t = \beta v_{t-1} + \nabla f(w_t)
$$
$$
w_{t+1} = w_t - \eta v_t
$$

动量可以加速高条件数问题的收敛，减少震荡。

In [ ]:
# 优化中的线性代数：梯度下降 vs 动量
# 二次型 f(w) = 0.5 * w^T H w，H 是病态正定矩阵
H = torch.tensor([[1.0, 0.0], [0.0, 20.0]])  # 条件数 = 20
w_star = torch.tensor([0.0, 0.0])  # 极小值

def f(w):
    return 0.5 * w @ H @ w

def grad_f(w):
    return H @ w

# 梯度下降
def sgd(w0, lr, n_steps):
    w = w0.clone()
    history = [w.clone()]
    for _ in range(n_steps):
        w = w - lr * grad_f(w)
        history.append(w.clone())
    return torch.stack(history)

# 动量
def momentum(w0, lr, beta, n_steps):
    w = w0.clone()
    v = torch.zeros_like(w)
    history = [w.clone()]
    for _ in range(n_steps):
        v = beta * v + grad_f(w)
        w = w - lr * v
        history.append(w.clone())
    return torch.stack(history)

w0 = torch.tensor([2.0, 1.0])
lr = 0.04
n_steps = 100

hist_sgd = sgd(w0, lr, n_steps)
hist_mom = momentum(w0, lr, 0.9, n_steps)

print(f"Hessian H =\n{H}")
print(f"条件数 κ = λ_max/λ_min = {20.0/1.0}")
print(f"学习率 lr = {lr}")

# 收敛曲线
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 等高线 + 路径
ax = axes[0]
w1 = torch.linspace(-2.5, 2.5, 100)
w2 = torch.linspace(-1.5, 1.5, 100)
W1, W2 = torch.meshgrid(w1, w2, indexing='ij')
F_val = 0.5 * (H[0,0]*W1**2 + H[1,1]*W2**2)
cs = ax.contour(W1.numpy(), W2.numpy(), F_val.numpy(), levels=15, cmap='RdYlBu')
ax.plot(hist_sgd[:, 0].numpy(), hist_sgd[:, 1].numpy(), 'b-o', markersize=2, linewidth=1, alpha=0.7, label='SGD')
ax.plot(hist_mom[:, 0].numpy(), hist_mom[:, 1].numpy(), 'r-o', markersize=2, linewidth=1, alpha=0.7, label='Momentum')
ax.scatter([0], [0], c='black', s=100, marker='*', zorder=5, label='极小值')
ax.set_title("优化路径：SGD (锯齿) vs Momentum (平滑)")
ax.set_xlabel("w1"); ax.set_ylabel("w2")
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# 收敛曲线
ax = axes[1]
error_sgd = torch.norm(hist_sgd, dim=1)
error_mom = torch.norm(hist_mom, dim=1)
ax.plot(error_sgd.numpy(), 'b-', linewidth=2, label='SGD')
ax.plot(error_mom.numpy(), 'r-', linewidth=2, label='Momentum (β=0.9)')
ax.set_xlabel("迭代步数"); ax.set_ylabel("||w - w*||")
ax.set_title("收敛曲线：Momentum 收敛更快")
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.suptitle(f"优化中的线性代数：条件数 κ={20} 对收敛的影响", fontsize=13)
plt.tight_layout()
plt.show()

print(f"\n最终误差 (100步后):")
print(f"  SGD:      {error_sgd[-1].item():.6f}")
print(f"  Momentum: {error_mom[-1].item():.6f}")
print(f"→ 动量通过积累梯度方向，减少高曲率方向的震荡，加速收敛")

## 7. 降维与表示学习（PCA / 词嵌入几何）

### 7.1 PCA = SVD 的应用

主成分分析（PCA）找到数据方差最大的方向，用于降维：

1. 数据中心化：$X_c = X - \mu$
2. SVD 分解：$X_c = U \Sigma V^T$
3. 主成分方向 = V 的列（即 $X_c^T X_c$ 的特征向量）
4. 降维：取前 k 个主成分，$Y = X_c V[:, :k]$

**与第 4 节的联系**：PCA 就是数据矩阵的截断 SVD / 低秩近似。

### 7.2 词嵌入的几何

词向量（Word2Vec / GloVe）将词映射到低维向量空间。线性代数性质：
- 相似词在向量空间中距离近（余弦相似度高）
- 类比关系：$v_{king} - v_{man} + v_{woman} \approx v_{queen}$
- 语义方向可以通过向量运算捕捉

In [ ]:
# PCA 手动实现：用 SVD 对数据降维
torch.manual_seed(42)
n_samples = 200
# 生成 3D 数据，但内在维度是 2（加噪声）
Z = torch.randn(n_samples, 2) @ torch.tensor([[2.0, 1.0, 0.5], [0.5, 1.5, 2.0]])
X = Z + 0.1 * torch.randn(n_samples, 3)  # 3D 数据

print(f"数据 X: {X.shape}")
print(f"数据的秩: {torch.linalg.matrix_rank(X).item()}")

# PCA 步骤
# 1. 中心化
mu = X.mean(dim=0)
X_c = X - mu

# 2. SVD
U, S, Vh = torch.linalg.svd(X_c, full_matrices=False)
print(f"\n奇异值: {S.tolist()}")
print(f"方差解释比: {(S**2 / (S**2).sum() * 100).tolist()}")

# 3. 降维到 2D：Y = X_c @ V[:2, :].T
k = 2
V = Vh.T  # [3, 3]
Y = X_c @ V[:, :k]  # [200, 2]
print(f"\n降维后 Y: {Y.shape}")

# 4. 重构（低秩近似）
X_reconstructed = Y @ V[:, :k].T + mu
recon_error = torch.norm(X - X_reconstructed) / torch.norm(X) * 100
print(f"重构相对误差: {recon_error:.2f}%")

# 可视化
fig = plt.figure(figsize=(14, 5))

# 原始 3D 数据
ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(X[:, 0], X[:, 1], X[:, 2], alpha=0.3, s=10, c='blue')
# 主成分方向
for i in range(3):
    v = V[:, i] * S[i] / n_samples**0.5
    ax1.quiver(mu[0], mu[1], mu[2], v[0], v[1], v[2], color='red', linewidth=2)
ax1.set_title("原始 3D 数据 + 主成分方向")

# 降维到 2D
ax2 = fig.add_subplot(132)
ax2.scatter(Y[:, 0], Y[:, 1], alpha=0.3, s=10, c='red')
ax2.set_title("PCA 降维到 2D")
ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)

# 方差解释
ax3 = fig.add_subplot(133)
cum_var = torch.cumsum(S**2, dim=0) / (S**2).sum() * 100
ax3.bar(range(1, 4), (S**2/(S**2).sum()*100).tolist(), color='steelblue', alpha=0.7)
ax3.plot(range(1, 4), cum_var.tolist(), 'ro-', linewidth=2, label='累积方差')
ax3.set_xlabel("主成分"); ax3.set_ylabel("方差解释 (%)")
ax3.set_title("方差解释比：前2个主成分 >99%")
ax3.legend(); ax3.grid(True, alpha=0.3)

plt.suptitle("PCA = SVD：数据降维与低秩近似", fontsize=13)
plt.tight_layout()
plt.show()

print("\n→ PCA 的本质：找到数据方差最大的方向（协方差矩阵的特征向量）")
print("→ 等价于数据矩阵的 SVD：主成分 = V 的列，方差 = 奇异值平方")
print("→ 降维 = 截断 SVD = 最佳低秩近似（Eckart-Young 定理）")

In [ ]:
# 词嵌入的线性代数性质：类比关系
# 用一个简化的例子演示词向量的几何性质
# 假设我们有以下词向量（从训练好的模型中简化）
words = {
    'king':    torch.tensor([0.8, 0.6, 0.1, 0.3]),
    'queen':   torch.tensor([0.7, 0.5, 0.8, 0.2]),
    'man':     torch.tensor([0.2, 0.9, 0.1, 0.4]),
    'woman':   torch.tensor([0.1, 0.8, 0.9, 0.3]),
    'prince':  torch.tensor([0.7, 0.5, 0.2, 0.8]),
    'princess':torch.tensor([0.6, 0.4, 0.8, 0.7]),
    'dog':     torch.tensor([0.1, 0.2, 0.1, 0.1]),
    'cat':     torch.tensor([0.1, 0.2, 0.1, 0.05]),
}

def cosine_sim(a, b):
    return torch.dot(a, b) / (torch.norm(a) * torch.norm(b))

print("=== 词向量的余弦相似度 ===")
print(f"{'词对':>20} {'余弦相似度':>10}")
print("-" * 35)
pairs = [('king', 'queen'), ('man', 'woman'), ('king', 'man'), ('queen', 'woman'),
         ('prince', 'princess'), ('dog', 'cat'), ('king', 'dog'), ('queen', 'cat')]
for w1, w2 in pairs:
    sim = cosine_sim(words[w1], words[w2]).item()
    print(f"{w1+' - '+w2:>20} {sim:>10.4f}")

print("\n=== 类比关系：king - man + woman ≈ queen ===")
analogy = words['king'] - words['man'] + words['woman']
print(f"king - man + woman = {analogy.tolist()}")
print(f"与 queen 的余弦相似度: {cosine_sim(analogy, words['queen']).item():.4f}")

# 找最接近的词
print("\n与类比向量最接近的词:")
sims = [(w, cosine_sim(analogy, v).item()) for w, v in words.items()]
sims.sort(key=lambda x: -x[1])
for w, s in sims[:3]:
    print(f"  {w}: {s:.4f}")

print("\n=== 另一个类比：prince - man + woman ≈ princess ===")
analogy2 = words['prince'] - words['man'] + words['woman']
print(f"与 princess 的余弦相似度: {cosine_sim(analogy2, words['princess']).item():.4f}")

print("\n→ 词嵌入将语义关系编码为向量空间中的线性运算")
print("→ '性别' 方向 ≈ woman - man，'王室' 方向 ≈ king - man")
print("→ 类比 = 向量加减法，这是线性代数在 NLP 中的优美体现")

# 可视化：用 PCA 把 4D 词向量降到 2D
X_words = torch.stack(list(words.values()))
mu_w = X_words.mean(dim=0)
X_wc = X_words - mu_w
_, _, Vh_w = torch.linalg.svd(X_wc, full_matrices=False)
Y_words = X_wc @ Vh_w[:2, :].T

fig, ax = plt.subplots(figsize=(8, 6))
for (w, _), y in zip(words.items(), Y_words):
    ax.scatter(y[0], y[1], s=100, zorder=5)
    ax.annotate(w, (y[0], y[1]), textcoords="offset points", xytext=(5, 5), fontsize=11)
# 画类比箭头
ax.annotate('', xy=Y_words[1], xytext=Y_words[0], 
            arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.set_title("词向量的 2D 可视化（PCA 降维）")
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 8. 数值稳定性与计算效率

### 8.1 条件数与数值稳定性

线性方程组 $Ax = b$ 的解对扰动的敏感性由条件数 $\kappa(A) = \|A\| \|A^{-1}\|$ 决定：

- $\kappa \approx 1$：良态，解稳定
- $\kappa \gg 1$：病态，b 的微小扰动导致解的巨大变化

### 8.2 深度学习中的数值稳定性

- **梯度消失/爆炸**：RNN 中矩阵多次相乘，特征值 >1 导致爆炸，<1 导致消失
- **Softmax 溢出**：输入值过大时 $e^x$ 溢出，需要减去最大值
- **矩阵求逆不稳定**：避免直接求逆，用 solve / lstsq
- **混合精度训练**：FP16 范围有限，需要梯度缩放（loss scaling）

### 8.3 计算效率

- 矩阵乘法复杂度 $O(mnp)$，是深度学习的计算瓶颈
- GPU 专为矩阵乘法优化（Tensor Core）
- 卷积通过 im2col 转化为矩阵乘法，复用 GEMM 优化
- 注意力的 $O(N^2)$ 复杂度是长序列的瓶颈

In [ ]:
# 条件数对线性方程组解的稳定性影响
# 构造一个病态矩阵（希尔伯特矩阵型）
n = 5
A = torch.zeros(n, n)
for i in range(n):
    for j in range(n):
        A[i, j] = 1.0 / (i + j + 1)  # 希尔伯特矩阵

x_true = torch.ones(n)
b = A @ x_true

print("希尔伯特矩阵 H (5x5):")
print(A)
print(f"\n条件数 κ(H) = {torch.linalg.cond(A).item():.2e} (非常大!)")

# 给 b 加微小扰动
b_perturbed = b + 1e-6 * torch.randn(n)
x_perturbed = torch.linalg.solve(A, b_perturbed)

print(f"\nb 的扰动大小: {torch.norm(b_perturbed - b).item():.2e}")
print(f"解的扰动大小: {torch.norm(x_perturbed - x_true).item():.2e}")
print(f"放大倍数: {torch.norm(x_perturbed - x_true).item() / torch.norm(b_perturbed - b).item():.2e}")
print(f"≈ 条件数 κ = {torch.linalg.cond(A).item():.2e}")

print("\n→ 病态矩阵：b 的微小扰动被条件数放大，解完全不可靠")
print("→ 实际中避免直接求逆，用 solve/lstsq，并关注条件数")

In [ ]:
# Softmax 的数值稳定性：减去最大值防止溢出
print("=== Softmax 数值稳定性 ===")
x_large = torch.tensor([1000.0, 1001.0, 1002.0])

# 直接计算（会溢出）
print(f"输入: {x_large.tolist()}")
try:
    exp_x = torch.exp(x_large)
    print(f"exp(x) = {exp_x.tolist()} (溢出为 inf!)")
    softmax_direct = exp_x / exp_x.sum()
    print(f"直接 softmax = {softmax_direct.tolist()}")
except Exception as e:
    print(f"直接计算报错: {e}")

# 稳定计算：减去最大值
x_max = x_large.max()
x_shifted = x_large - x_max
exp_shifted = torch.exp(x_shifted)
softmax_stable = exp_shifted / exp_shifted.sum()
print(f"\n减去最大值后: {x_shifted.tolist()}")
print(f"exp(x - max) = {exp_shifted.tolist()}")
print(f"稳定 softmax = {softmax_stable.tolist()}")

# 验证：与 PyTorch 一致
print(f"\nPyTorch softmax = {F.softmax(x_large, dim=0).tolist()}")
print(f"一致? {torch.allclose(softmax_stable, F.softmax(x_large, dim=0))}")

print("\n→ softmax(x) = softmax(x - c) 对任意常数 c 成立")
print("→ 减去最大值保证所有 exp 输入 ≤ 0，不会溢出")
print("→ 这是深度学习中最基本的数值稳定技巧")

In [ ]:
# 矩阵多次相乘：梯度消失/爆炸的线性代数解释
print("=== 矩阵多次相乘与梯度消失/爆炸 ===")
torch.manual_seed(42)

# 情况1：特征值 > 1 → 爆炸
A_explode = torch.tensor([[1.5, 0.0], [0.0, 1.2]])
# 情况2：特征值 < 1 → 消失
A_vanish = torch.tensor([[0.5, 0.0], [0.0, 0.8]])
# 情况3：正交矩阵（特征值模=1）→ 保持
A_orth = torch.linalg.qr(torch.randn(2, 2))[0]

x0 = torch.tensor([1.0, 1.0])
steps = 20

norms_explode = [torch.norm(x0).item()]
norms_vanish = [torch.norm(x0).item()]
norms_orth = [torch.norm(x0).item()]

x_e, x_v, x_o = x0.clone(), x0.clone(), x0.clone()
for _ in range(steps):
    x_e = A_explode @ x_e
    x_v = A_vanish @ x_v
    x_o = A_orth @ x_o
    norms_explode.append(torch.norm(x_e).item())
    norms_vanish.append(torch.norm(x_v).item())
    norms_orth.append(torch.norm(x_o).item())

print(f"爆炸矩阵特征值: {torch.linalg.eigvalsh(A_explode).tolist()} (全>1)")
print(f"消失矩阵特征值: {torch.linalg.eigvalsh(A_vanish).tolist()} (全<1)")
print(f"正交矩阵特征值模: {torch.linalg.eigvals(A_orth).abs().tolist()} (=1)")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(norms_explode, 'r-', linewidth=2, label='特征值>1 (爆炸)')
ax.plot(norms_vanish, 'b-', linewidth=2, label='特征值<1 (消失)')
ax.plot(norms_orth, 'g-', linewidth=2, label='正交矩阵 (保持)')
ax.set_xlabel("相乘次数"); ax.set_ylabel("||A^k x||")
ax.set_title("矩阵多次相乘：特征值决定范数增长/衰减")
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print("\n→ RNN 中 h_t = W h_{t-1}，多次相乘后梯度由 W 的特征值谱决定")
print("→ 特征值 > 1：梯度爆炸（需要梯度裁剪）")
print("→ 特征值 < 1：梯度消失（LSTM/GRU/残差连接解决）")
print("→ 正交初始化（特征值模=1）：缓解梯度问题")

## 总结：线性代数是深度学习的骨架

回顾整个课程，线性代数贯穿深度学习的每一个角落：

| 概念 | 深度学习中的体现 |
|------|----------------|
| 向量 | 数据样本、特征、词嵌入、梯度 |
| 矩阵乘法 | 全连接层、注意力、状态转移 |
| 特征值/特征向量 | PCA、谱归一化、初始化、优化曲率 |
| SVD | 降维、低秩近似、伪逆、推荐系统 |
| 投影/最小二乘 | 线性回归、岭回归、数据拟合 |
| 正定矩阵 | Hessian、协方差矩阵、马氏距离、高斯分布 |
| 条件数 | 优化收敛速度、数值稳定性、梯度消失/爆炸 |
| 正交性 | 正交初始化、Gram-Schmidt、注意力归一化 |
| 秩 | 数据内在维度、模型压缩、低秩适应 |

**核心洞察**：深度学习模型 = 一系列线性变换（矩阵乘法）+ 非线性激活的交替。理解了线性代数，就理解了模型的行为边界、优化的动力学、以及数值稳定性的根源。

下一站：概率论与信息论——从确定性的线性变换，走向不确定性的建模。

## 课后练习

### 全连接层

1. 手动实现一个全连接层的前向和反向传播（不用 `nn.Linear`），输入维度 10，输出维度 5，batch size 8。验证你的反向传播与 PyTorch autograd 结果一致。

2. 计算一个 3 层 MLP（784→256→128→10）的总参数量和每层的乘法次数（batch size=32），分析哪一层参数最多、计算量最大。

### 卷积

3. 用 im2col 方法手动实现 2D 卷积（输入 1×1×5×5，卷积核 1×1×3×3，stride=1，padding=0），验证结果与 `F.conv2d` 一致。

4. 证明 1D 卷积（valid mode）可以表示为 Toeplitz 矩阵与向量的乘法。用一个长度为 4 的信号和长度为 2 的卷积核验证。

### 注意力

5. 手动实现多头注意力（d_model=8，n_heads=2，序列长度=6），包括 Q/K/V 线性投影、缩放点积注意力、多头拼接、输出投影。验证与 `F.multi_head_attention_forward` 结果一致。

6. 实验验证缩放的重要性：对 d_k = 4, 16, 64, 256，分别计算不缩放和缩放后的注意力权重的最大熵，画出 d_k 与最大注意力权重的关系曲线，解释为什么需要除以 $\sqrt{d_k}$。

### 优化

7. 构造一个条件数为 100 的 2D 二次型，分别用 SGD 和动量（β=0.9）优化，画出优化路径和收敛曲线，解释动量为什么能加速收敛。

8. 梯度消失/爆炸实验：构造三个 2×2 矩阵（特征值分别为 0.5、1.0、2.0），分别与初始向量相乘 30 次，画出范数变化曲线，解释 RNN 中梯度问题的根源。

### 降维与表示

9. 用 PCA 对一个 100×10 的随机数据矩阵（内在维度 3，加噪声）降维到 3D，计算重构误差，画出奇异值衰减曲线和累积方差解释曲线。

10. 词嵌入类比实验：构造 8 个词的简化词向量，验证 "king - man + woman ≈ queen" 和 "prince - man + woman ≈ princess"，用余弦相似度量化。

### 数值稳定性

11. 构造一个 6×6 希尔伯特矩阵，给 b 加 1e-8 的扰动，比较解的变化，计算放大倍数并与条件数对比。

12. 实现数值稳定的 softmax（减去最大值），对输入 [1000, 1001, 1002] 验证不会溢出，并解释为什么 softmax(x) = softmax(x - c)。

### 综合题

13. 选择一个你熟悉的深度学习模型（如 MLP、CNN、Transformer），列出其中所有用到线性代数的组件，并说明每个组件对应课程中的哪个概念。

14. 用自己的话解释：为什么说"深度学习 = 线性变换 + 非线性激活的交替堆叠"？非线性激活的作用是什么？如果去掉所有非线性激活，模型会变成什么？